### TESTING

In [ ]:
train_dataset = VideoFrameDataset("dataset_split.csv", split="train")
label_to_idx = train_dataset.label_to_idx  # {label_str: idx_int}
idx_to_label = {v: k for k, v in label_to_idx.items()}


In [2]:
import cv2
import torch
import numpy as np
import torch.nn.functional as F
import torchvision.transforms as transforms

def predict_video(
    model,
    video_path,
    idx_to_label,
    frames_per_video=120,
    resize_height=224,
    resize_width=224,
    device='cpu'
):
    """
    Predicts the label (and probability distribution) of a single video .npy file
    using the trained 2D CNN + LSTM model.

    Args:
        model           : A CNNLSTMModel instance (in eval mode).
        video_path      : Path to the .npy file of shape (T, H, W, 3).
        idx_to_label    : dict mapping integer class indices -> label strings.
        frames_per_video: The fixed number of frames (pad or truncate to this).
        resize_height   : Height for frame resize (e.g., 224).
        resize_width    : Width for frame resize (e.g., 224).
        device          : 'cpu' or 'cuda'.

    Returns:
        pred_label (str)                 : The predicted label name.
        prob_dict (dict[label] -> float) : Probability for each label, if you want the distribution.
    """
    # 1. Load the .npy frames
    frames = np.load(video_path)  # shape (T, H, W, 3)
    T = frames.shape[0]

    # 2. Pad or truncate to frames_per_video if needed
    if T > frames_per_video:
        frames = frames[:frames_per_video]
    elif T < frames_per_video:
        pad_count = frames_per_video - T
        pad_shape = (pad_count,) + frames.shape[1:]
        pad_array = np.zeros(pad_shape, dtype=frames.dtype)
        frames = np.concatenate([frames, pad_array], axis=0)

    # 3. Apply the same transforms used in training
    transform_pipeline = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((resize_height, resize_width)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    transformed_frames = []
    for frame in frames:
        # If your frames are BGR, you need to convert:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        tensor_frame = transform_pipeline(frame_rgb)  # shape (3, 224, 224)
        transformed_frames.append(tensor_frame)

    # 4. Stack into shape (T, 3, 224, 224), then unsqueeze for batch dimension -> (1, T, 3, 224, 224)
    video_tensor = torch.stack(transformed_frames, dim=0).unsqueeze(0).to(device)

    # 5. Forward pass
    with torch.no_grad():
        logits = model(video_tensor)  # shape (1, num_classes)

    # 6. Softmax to get probabilities
    probs = F.softmax(logits, dim=1)  # shape (1, num_classes)

    # 7. Argmax for predicted label index
    _, pred_idx = torch.max(probs, dim=1)  # shape (1,)
    pred_idx = pred_idx.item()

    pred_label = idx_to_label[pred_idx]

    # 8. Build probability dictionary, if desired
    probs_np = probs.squeeze(0).cpu().numpy()
    prob_dict = {}
    for class_idx, p in enumerate(probs_np):
        class_label = idx_to_label[class_idx]
        prob_dict[class_label] = float(p)  # convert to plain Python float

    return pred_label, prob_dict
